# 🏆 08. PageRank & Phân tích Vai trò Tướng (Core Carry vs Flex Support)

**Thành viên thực hiện:** Thành viên 2 (Graph Mining Lead)

Mục tiêu:
- Ứng dụng **PageRank** trên đồ thị liên kết tướng để xếp hạng tầm quan trọng của từng nhân vật.
- Đối chiếu PageRank với Win-rate đơn thuần để phân loại: Core Carry, Flex Support, Overrated, Hidden Gem.
- Xác định nhân vật hạt nhân của từng Archetype đã tìm được ở bước phân cụm.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from src.graph_miner import GraphMiner
import warnings
warnings.filterwarnings('ignore')

## 1. Nạp dữ liệu và Đồ thị

In [10]:
cooc_wr = pd.read_pickle('../data/processed/cooc_winrate.pkl')
df_processed = pd.read_csv('../data/processed/tft_processed.csv')
gm = GraphMiner()
G = gm.build_network(cooc_wr, min_weight=0.55)

📊 Network: 107 nodes, 1447 edges, density=0.2552


## 2. Tính toán PageRank với các Damping Factor khác nhau

In [11]:
pr_85, ranking_85 = gm.compute_pagerank(G, alpha=0.85)
pr_50, _ = gm.compute_pagerank(G, alpha=0.50)
pr_95, _ = gm.compute_pagerank(G, alpha=0.95)

pr_compare = pd.DataFrame({
    'alpha_0.50': pr_50,
    'alpha_0.85': pr_85,
    'alpha_0.95': pr_95
})
pr_compare.sort_values(by='alpha_0.85', ascending=False).head(15)

🏆 PageRank Top 5 (α=0.85):
   Thresh: 0.0230
   Soraka: 0.0218
   Nami: 0.0216
   Lulu: 0.0205
   MissFortune: 0.0183
🏆 PageRank Top 5 (α=0.5):
   Thresh: 0.0199
   Lulu: 0.0194
   Soraka: 0.0179
   MissFortune: 0.0172
   Ekko: 0.0171
🏆 PageRank Top 5 (α=0.95):
   Nami: 0.0238
   Thresh: 0.0224
   Soraka: 0.0222
   Taric: 0.0190
   Lulu: 0.0188


,alpha_0.50,alpha_0.85,alpha_0.95
Thresh,0.019919,0.023020,0.022438
Soraka,0.017857,0.021828,0.022177
Nami,0.016314,0.021575,0.023816
Lulu,0.019367,0.020467,0.018846
MissFortune,0.017194,0.018273,0.016961
Ekko,0.017060,0.018135,0.016843
Taric,0.013709,0.017327,0.019001
Amumu,0.013172,0.016857,0.018592
LuxElectric,0.013283,0.016727,0.018347
MasterYi,0.013497,0.016691,0.018140


## 3. Đối chiếu PageRank vs Win-rate (Ma trận 4 góc chiến thuật)

In [12]:
centrality = gm.compute_centrality_metrics(G)
centrality['pagerank'] = centrality.index.map(pr_85)

# Trực quan hóa ma trận PageRank vs Betweenness Centrality
fig, ax = plt.subplots(figsize=(12, 7))
sns.scatterplot(data=centrality, x='betweenness', y='pagerank', size='degree', sizes=(50, 400), hue='pagerank', palette='flare', ax=ax)

median_bet = centrality['betweenness'].median()
median_pr = centrality['pagerank'].median()
ax.axvline(median_bet, color='gray', linestyle='--')
ax.axhline(median_pr, color='gray', linestyle='--')

for node, row in centrality.iterrows():
    if row['pagerank'] > centrality['pagerank'].quantile(0.8) or row['betweenness'] > centrality['betweenness'].quantile(0.8):
        ax.text(row['betweenness'] + 0.001, row['pagerank'], node, fontsize=9)

ax.set_title('Ma trận Phân loại Tướng: PageRank (Tầm quan trọng) vs Betweenness (Tính bắc cầu Flex)')
ax.set_xlabel('Betweenness Centrality (Cầu nối giữa các đội hình)')
ax.set_ylabel('PageRank Score (Nhân vật hạt nhân / Core Carry)')
plt.show()

## 4. Xác định Core Carry của từng Đội hình

In [13]:
part_louvain, _, _ = gm.detect_communities_louvain(G)
from src.champion_metadata import CHAMPIONS_SET3 as CHAMPIONS
archetypes = gm.identify_meta_archetypes(G, part_louvain, champion_data=CHAMPIONS)
display(archetypes[['community', 'archetype', 'core_champion', 'core_pagerank', 'size']])

🔵 Louvain (res=1.0): 2 communities, modularity=0.3043


,community,archetype,core_champion,core_pagerank,size
0,0,Rebel + Blaster,Nami,0.0238,63
1,1,DarkStar + StarGuardian,Thresh,0.0224,44


## 5. Kết luận PageRank
- **PageRank cao + Betweenness thấp**: Tướng Core Carry chuyên biệt (Carry gánh team của 1 đội hình cụ thể).
- **PageRank cao + Betweenness cao**: Tướng Flex quốc dân (vừa mạnh vừa ghép được vào nhiều đội hình).